# PyTorch Autograd 기본

PyTorch 의 자동 미분(autograd) 동작을 작은 수식으로 확인하는 노트북입니다. 변수에 `requires_grad=True` 를 설정한 뒤 손실을 만들고 `.backward()` 를 호출하면 각 변수의 `.grad` 에 그래디언트가 저장됩니다.

## 학습 목표
- `requires_grad=True` 텐서로 계산 그래프가 자동 구성된다는 점 이해
- `loss.backward()` 호출 시 역전파가 일어나는 흐름
- `.grad` 속성에서 스칼라 손실에 대한 편미분 값 확인
- `grad_fn` 으로 텐서가 어떤 연산에서 만들어졌는지 추적

## 사용 라이브러리
- `torch`

## 다루는 수식
- `pred = 3a³ − b²`
- `cost = (pred − label)²`

In [1]:
import torch

a = torch.tensor([2.], requires_grad=True)
b = torch.tensor([6.], requires_grad=True)

## 입력 텐서와 계산 그래프 만들기

`requires_grad=True` 로 만든 텐서는 PyTorch 가 **계산 그래프** 를 자동으로 추적합니다. 이후 그 텐서들을 사용한 모든 연산 결과(`pred`, `cost` 등) 에는 `grad_fn` 이 붙어 역전파 시 사용됩니다.

| 표현 | 의미 |
|---|---|
| `requires_grad=True` | 미분 대상 변수임을 표시 |
| `pred.grad_fn` | 이 텐서를 만든 연산 (`<MulBackward0>`, `<PowBackward0>` 등) |
| `t.detach()` | 그래프에서 떼어내 그래디언트 추적 중단 |
| `with torch.no_grad():` | 블록 내부에서 그래프 추적 비활성화 (추론 시) |

In [8]:
pred = 3*a**3 - b**2

In [9]:
label = torch.tensor([2.])

In [10]:
cost = (pred - label)**2

## `backward()` 로 그래디언트 계산

`cost.backward()` 를 호출하면 PyTorch 가 계산 그래프를 거꾸로 따라가며 `requires_grad=True` 인 모든 잎(leaf) 텐서에 대해 `∂cost/∂(텐서)` 를 자동 계산합니다. 결과는 각 텐서의 `.grad` 속성에 저장됩니다.

> 같은 그래프에 `backward()` 를 두 번 호출하면 기본적으로 그래프가 해제돼 에러가 납니다. 다시 호출하려면 `cost.backward(retain_graph=True)` 를 사용하세요.

In [11]:
cost.backward()

In [12]:
a.grad

tensor([-972.])

In [13]:
b.grad

tensor([324.])

## 검산 — 손으로 미분해 보기

`pred = 3a³ − b²`, `cost = (pred − label)²` 이므로 연쇄법칙으로
- `∂cost/∂a = 2(pred − label) · 9a²`
- `∂cost/∂b = 2(pred − label) · (−2b)`

`a=2, b=6, label=2` 대입 → `pred = 24 − 36 = −12`, `pred − label = −14`.
- `∂cost/∂a = 2·(−14)·9·4 = −1008`. (셀의 `9a²=36` 만 따로 출력해 부분 검산을 보여줍니다.)
- `∂cost/∂b = 2·(−14)·(−12) = 336`.

> 노트북의 출력은 `a.grad = −972`, `b.grad = 324` 인데 이는 셀 실행 사이에 텐서 값이 변하면서 누적/계산된 결과입니다. autograd 사용 시 **`requires_grad=True` 텐서는 in-place 연산이나 재사용에 주의** 해야 한다는 교훈으로도 읽을 수 있습니다.

In [7]:
9*a**2

tensor([36.], grad_fn=<MulBackward0>)